#Mount Google Drive Untuk Dataset




In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#Install & Import Library

In [ ]:
!pip -q install timm

import os, random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import timm

#Path Dataset

In [ ]:
# Ganti sesuai lokasi di Drive kamu
BASE_PATH = "/content/drive/MyDrive/ZERO WASTE/FILE_SSI"

In [ ]:
import os

# Check if the BASE_PATH exists
if not os.path.exists(BASE_PATH):
    print(f"Error: BASE_PATH '{BASE_PATH}' does not exist. Please check your Google Drive path.")
else:
    print(f"Contents of BASE_PATH '{BASE_PATH}':")
    try:
        for item in os.listdir(BASE_PATH):
            print(f"- {item}")
    except Exception as e:
        print(f"Could not list contents, perhaps permission issue: {e}")


#30 Kelas menjadi 6 Kelas



In [ ]:
mapping_material = {
    # plastic
    "plastic_detergent_bottles": "plastic",
    "plastic_food_containers": "plastic",
    "plastic_trash_bags": "plastic",
    "plastic_straws": "plastic",
    "plastic_cup_lids": "plastic",
    "plastic_water_bottles": "plastic",
    "plastic_soda_bottles": "plastic",
    "plastic_shopping_bags": "plastic",
    "disposable_plastic_cutlery": "plastic",
    "styrofoam_cups": "plastic",
    "styrofoam_food_containers": "plastic",

    # paper
    "cardboard_boxes": "paper",
    "cardboard_packaging": "paper",
    "newspaper": "paper",
    "magazines": "paper",
    "office_paper": "paper",
    "paper_cups": "paper",

    # glass
    "glass_beverage_bottles": "glass",
    "glass_food_jars": "glass",
    "glass_cosmetic_containers": "glass",

    # metal
    "aluminum_food_cans": "metal",
    "aluminum_soda_cans": "metal",
    "steel_food_cans": "metal",
    "aerosol_cans": "metal",

    # organic
    "food_waste": "organic",
    "eggshells": "organic",
    "coffee_grounds": "organic",
    "tea_bags": "organic",

    # textile
    "clothing": "textile",
    "shoes": "textile",
}

In [ ]:
import random
from PIL import Image
import matplotlib.pyplot as plt
import os as original_os # Alias the os module to avoid any shadowing/redefinition issues

def display_default_samples_per_mapped_class(base_path, mapping_material, target_classes):
    plt.figure(figsize=(15, 3)) # Adjusted figure size for 6 images in a row

    # Group original folders by mapped class
    mapped_class_to_original_folders = {mapped_class: [] for mapped_class in target_classes}
    for original_folder, mapped_class in mapping_material.items():
        if mapped_class in target_classes:
            mapped_class_to_original_folders[mapped_class].append(original_folder)

    # Collect one random default image path for each mapped class
    samples_to_display = {}
    for mapped_class in target_classes:
        original_folders = mapped_class_to_original_folders.get(mapped_class, [])
        all_default_images_for_class = []

        for folder in original_folders:
            default_path = original_os.path.join(base_path, folder, 'default')
            if original_os.path.exists(default_path):
                # Only add if it's a file and a common image extension
                default_images = [original_os.path.join(default_path, f) for f in original_os.listdir(default_path) if original_os.path.isfile(original_os.path.join(default_path, f)) and f.lower().endswith(('.png', '.jpg', '.jpeg'))]
                all_default_images_for_class.extend(default_images)

        if all_default_images_for_class:
            samples_to_display[mapped_class] = random.choice(all_default_images_for_class)
        else:
            samples_to_display[mapped_class] = None

    # Display one default sample for each mapped class
    for i, mapped_class in enumerate(target_classes):
        plt.subplot(1, len(target_classes), i + 1)
        sample_path = samples_to_display.get(mapped_class)

        if sample_path:
            img = Image.open(sample_path).convert("RGB")
            plt.imshow(img)
            # Extract the original folder name for the title
            original_folder_name = original_os.path.basename(original_os.path.dirname(original_os.path.dirname(sample_path)))
            plt.title(f"{mapped_class.capitalize()} (Default)\n({original_folder_name})", fontsize=8)
        else:
            plt.text(0.5, 0.5, f"No default sample for {mapped_class}", ha='center', va='center', fontsize=8)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

# Define classes based on the unique values in mapping_material
# Assuming 'classes' variable is already defined from a previous cell (e.g., from cell 0377fba7)
# If not, uncomment the line below:
classes = sorted(list(set(mapping_material.values())))

print("Sample default images, one per mapped class:")
display_default_samples_per_mapped_class(BASE_PATH, mapping_material, classes)


#Preprocessing

In [ ]:
# Preprocessing + augmentation (hanya untuk train)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),         # samakan ukuran (lebih cepat)
    transforms.RandomHorizontalFlip(),     # augmentasi
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

# Untuk test (tanpa augmentasi)
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

# Untuk validation (tanpa augmentasi, sama dengan test_transform)
validation_transform = test_transform

#Load Dataset & Check Jumlah

In [ ]:
class WasteDataset(Dataset):
    def __init__(self, base_path, mode="default", transform=None):
        self.data = []
        self.labels = []
        self.transform = transform

        for folder in os.listdir(base_path):
            if folder in mapping_material:
                material = mapping_material[folder]
                subfolder = os.path.join(base_path, folder, mode)

                if os.path.exists(subfolder):
                    for img_name in os.listdir(subfolder):
                        self.data.append(os.path.join(subfolder, img_name))
                        self.labels.append(material)

        # mapping label → angka
        self.classes = sorted(list(set(self.labels)))
        self.class_to_idx = {c:i for i,c in enumerate(self.classes)}

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img = Image.open(self.data[idx]).convert("RGB")
        label = self.class_to_idx[self.labels[idx]]

        if self.transform:
            img = self.transform(img)

        return img, label

In [ ]:
from torch.utils.data import random_split, ConcatDataset, Dataset as BaseDataset

# Helper class to apply a specific transform to a Subset or a base Dataset
# This is necessary because random_split returns Subset objects which share
# the transform of their original dataset. We need to apply different transforms
# (train_transform for training data, test_transform for testing data) after splitting.
class TransformedSubset(BaseDataset):
    def __init__(self, subset_or_dataset, transform=None):
        if isinstance(subset_or_dataset, torch.utils.data.Subset):
            # If it's already a Subset from random_split
            self.dataset = subset_or_dataset.dataset # Reference to the original full dataset (e.g., WasteDataset)
            self.indices = subset_or_dataset.indices # Indices specific to this subset
        else:
            # If it's a direct Dataset (e.g., WasteDataset before splitting)
            self.dataset = subset_or_dataset
            self.indices = list(range(len(subset_or_dataset))) # All indices of the dataset
        self.transform = transform

    def __getitem__(self, idx):
        # Map the subset index to the original dataset index
        original_idx = self.indices[idx]
        # Get item from the *original* dataset. We assume WasteDataset was created with transform=None
        # so it returns a PIL Image, which we then transform here.
        img, label = self.dataset[original_idx]

        if self.transform:
            img = self.transform(img)
        return img, label

    def __len__(self):
        return len(self.indices)

# 1. Create base WasteDataset instances *without* transforms initially.
#    Transforms will be applied by TransformedSubset after splitting.
#    Assume WasteDataset, BASE_PATH, train_transform, validation_transform, test_transform are defined in previous cells.
if 'WasteDataset' not in globals():
    raise RuntimeError("WasteDataset class not defined. Please run the cell defining WasteDataset (e.g., kvSfoF-eAQfh).")
if 'BASE_PATH' not in globals():
    raise RuntimeError("BASE_PATH not defined. Please run the cell defining BASE_PATH (e.g., fBQo6uNB_p5H).")
if 'train_transform' not in globals() or 'validation_transform' not in globals() or 'test_transform' not in globals():
    raise RuntimeError("Transforms (train_transform, validation_transform, test_transform) not defined. Please run cell GWhPmwnuAKDY.")

full_default_raw_dataset = WasteDataset(BASE_PATH, mode="default", transform=None)
full_real_world_raw_dataset = WasteDataset(BASE_PATH, mode="real_world", transform=None)

# --- Speed Optimization (IMPORTANT): Extract all labels into a global list ---
globals()['global_label_list'] = full_default_raw_dataset.labels + full_real_world_raw_dataset.labels
print(f"Global label list created with {len(globals()['global_label_list'])} entries.")
# --- End Speed Optimization ---

# 2. Define overall target sizes based on 80% train, 10% val, 10% test of total combined data.
total_combined_data_size = len(full_default_raw_dataset) + len(full_real_world_raw_dataset)
target_train_size = int(0.8 * total_combined_data_size)
target_val_size = int(0.1 * total_combined_data_size)
target_test_size = total_combined_data_size - target_train_size - target_val_size # Adjust for rounding

# Ensure target_test_size doesn't exceed available real_world data.
if target_test_size > len(full_real_world_raw_dataset):
    print(f"Warning: Target test size ({target_test_size}) is larger than available real_world data ({len(full_real_world_raw_dataset)}). Adjusting test set to use all real_world data.")
    actual_real_world_test_size = len(full_real_world_raw_dataset)
else:
    actual_real_world_test_size = target_test_size

# 3. Extract Test Set (Real-World Only):
real_world_remaining_size = len(full_real_world_raw_dataset) - actual_real_world_test_size
real_world_for_train_val_subset, raw_test_subset = random_split(
    full_real_world_raw_dataset, [real_world_remaining_size, actual_real_world_test_size]
)
globals()['test_dataset'] = TransformedSubset(raw_test_subset, transform=test_transform)

# 4. Create Combined Pool for Training and Validation:
# This pool will consist of all default data and the remaining real_world data.
train_val_raw_pool = ConcatDataset([full_default_raw_dataset, real_world_for_train_val_subset])

# 5. Split Training and Validation from the combined pool
# The sizes for train and val are adjusted to maintain their *proportion* (80:10 or 8:1)
# within the remaining `train_val_raw_pool`, which represents 90% of the original total.
train_val_pool_size = len(train_val_raw_pool)

# Calculate distribution within the train_val_pool
# To maintain 80% and 10% of the *original total* (excluding the real_world test part)
# it's equivalent to splitting the remaining (90% of total) in an 8:1 ratio.
adjusted_train_size_from_pool = int((target_train_size / (target_train_size + target_val_size)) * train_val_pool_size)
adjusted_val_size_from_pool = train_val_pool_size - adjusted_train_size_from_pool

raw_train_subset, raw_val_subset = random_split(
    train_val_raw_pool, [adjusted_train_size_from_pool, adjusted_val_size_from_pool]
)

# 6. Apply Transforms and Create Final Datasets:
train_dataset = TransformedSubset(raw_train_subset, transform=train_transform)
validation_dataset = TransformedSubset(raw_val_subset, transform=validation_transform)

# 7. Get classes from one of the raw datasets. They should be consistent.
classes = full_default_raw_dataset.classes # Make sure classes is globally available

# 8. Create DataLoaders with the new datasets and batch size
batch_size = 256 # Updated batch size from 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(validation_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
globals()['test_loader'] = DataLoader(globals()['test_dataset'], batch_size=batch_size, shuffle=False, num_workers=0) # Ensure test_loader is global

# 9. Print summary of the splits
print(f"Total data (default + real_world): {total_combined_data_size}")
print(f"Train dataset size (aiming for 80% of total combined): {len(train_dataset)}")
print(f"Validation dataset size (aiming for 10% of total combined): {len(validation_dataset)}")
print(f"Test dataset size (aiming for 10% of total combined, real-world only): {len(globals()['test_dataset'])}")
print(f"Classes: {classes}")
print(f"Batch size: {batch_size}")

In [ ]:
from collections import Counter
from torch.utils.data import ConcatDataset, Subset
import torch
import os

# Helper function to recursively find the base WasteDataset and its original indices
# relative to the full combined raw dataset (default + real_world).
def _resolve_to_global_indices(dataset):
    # Base case: WasteDataset
    if isinstance(dataset, globals().get('WasteDataset')):
        if 'full_default_raw_dataset' in globals() and dataset is globals()['full_default_raw_dataset']:
            return list(range(len(dataset))) # Indices for default dataset
        elif 'full_real_world_raw_dataset' in globals() and dataset is globals()['full_real_world_raw_dataset']:
            # Offset needed because global_label_list is default labels then real_world labels
            offset = len(globals()['full_default_raw_dataset']) if 'full_default_raw_dataset' in globals() else 0
            return [i + offset for i in range(len(dataset))]
        else:
            # This case should ideally not happen if only the two global raw datasets are used.
            # If WasteDataset was instantiated directly from within this cell, it might not be identified.
            # Fallback to assuming its indices are global within itself, which might be incorrect for a subset.
            print(f"Warning: Unidentified WasteDataset instance encountered: {dataset}. Proceeding with assumption.")
            return list(range(len(dataset)))

    # Recursive cases: TransformedSubset, Subset, ConcatDataset
    elif isinstance(dataset, globals().get('TransformedSubset')):
        # TransformedSubset.indices are relative to its internal .dataset
        resolved_base_indices = _resolve_to_global_indices(dataset.dataset)
        return [resolved_base_indices[i] for i in dataset.indices]

    elif isinstance(dataset, torch.utils.data.Subset):
        # Subset.indices are relative to its internal .dataset
        resolved_base_indices = _resolve_to_global_indices(dataset.dataset)
        return [resolved_base_indices[i] for i in dataset.indices]

    elif isinstance(dataset, ConcatDataset):
        all_global_indices = []
        for ds in dataset.datasets:
            all_global_indices.extend(_resolve_to_global_indices(ds))
        return all_global_indices

    else:
        raise TypeError(f"Unsupported dataset type for resolving global indices: {type(dataset)}")

# Modified _get_labels_from_dataset to leverage global_label_list
def _get_labels_from_dataset(dataset):
    if 'global_label_list' not in globals():
        print("Error: 'global_label_list' not found. Please ensure cell 'lfw8p9k1AVGN' has been executed or datasets re-initialized.")
        # Fallback to slow method if global_label_list isn't there
        return _get_labels_from_dataset_slow(dataset)

    # Direct return for WasteDataset if it's the base raw one
    if isinstance(dataset, globals().get('WasteDataset')):
        return dataset.labels

    try:
        # Attempt to get global indices and retrieve labels from global_label_list
        global_indices = _resolve_to_global_indices(dataset)
        return [globals()['global_label_list'][idx] for idx in global_indices]
    except (TypeError, ValueError) as e:
        print(f"Warning: Could not resolve global indices for {type(dataset)} ({e}). Falling back to slow label inference.")
        return _get_labels_from_dataset_slow(dataset)

# Original slow fallback method (kept for robustness, though should ideally be avoided)
def _get_labels_from_dataset_slow(dataset):
    if hasattr(dataset, 'labels'):
        return dataset.labels
    elif 'classes' in globals() and isinstance(globals()['classes'], list):
        inferred_labels = []
        try:
            for i in range(len(dataset)):
                _, label_idx = dataset[i]
                if isinstance(label_idx, int) and 0 <= label_idx < len(globals()['classes']):
                    inferred_labels.append(globals()['classes'][label_idx])
                else:
                    inferred_labels.append(str(label_idx))
            return inferred_labels
        except Exception as e:
            print(f"Warning: Could not infer labels from dataset __getitem__ (slow path): {e}")
            return []
    else:
        print(f"Warning: Unsupported dataset type and no global 'classes' for label inference (slow path): {type(dataset)}")
        return []

def summary_dataset(dataset, nama):
    print(f"\n=== {nama} ===")
    print("Total data:", len(dataset))

    labels = _get_labels_from_dataset(dataset)
    counter = Counter(labels)

    for k, v in counter.items():
        print(f"{k}: {v} gambar")

def summary_total_data(train_ds, val_ds, test_ds, nama="Combined Data"):
    print(f"\n=== {nama} ===")
    print("Total data:", len(train_ds) + len(val_ds) + len(test_ds))

    train_labels = _get_labels_from_dataset(train_ds)
    val_labels = _get_labels_from_dataset(val_ds)
    test_labels = _get_labels_from_dataset(test_ds)

    combined_labels_counter = Counter(train_labels)
    combined_labels_counter.update(val_labels)
    combined_labels_counter.update(test_labels)

    for k, v in combined_labels_counter.items():
        print(f"{k}: {v} gambar")

# Check if train_dataset, validation_dataset, and test_dataset are already defined.
# If not, attempt to define them using the logic from cell 'lfw8p9k1AVGN' (Load Dataset & Check Jumlah)
if 'train_dataset' not in globals() or 'validation_dataset' not in globals() or 'test_dataset' not in globals() or 'classes' not in globals():
    print("\n--- Re-initializing datasets and classes for robustness ---")
    from torch.utils.data import random_split, ConcatDataset, Dataset as BaseDataset

    # Helper class to apply a specific transform to a Subset or a base Dataset
    class TransformedSubset(BaseDataset):
        def __init__(self, subset_or_dataset, transform=None):
            if isinstance(subset_or_dataset, torch.utils.data.Subset):
                self.dataset = subset_or_dataset.dataset
                self.indices = subset_or_dataset.indices
            else:
                self.dataset = subset_or_dataset
                self.indices = list(range(len(subset_or_dataset)))
            self.transform = transform

        def __getitem__(self, idx):
            original_idx = self.indices[idx]
            img, label = self.dataset[original_idx]
            if self.transform:
                img = self.transform(img)
            return img, label

        def __len__(self):
            return len(self.indices)

    # Assuming WasteDataset, BASE_PATH, mapping_material, train_transform, validation_transform, test_transform are defined in previous cells
    if 'WasteDataset' not in globals():
        print("Error: WasteDataset class not found. Please run the cell defining WasteDataset (e.g., kvSfoF-eAQfh).")
    elif 'BASE_PATH' not in globals():
        print("Error: BASE_PATH not found. Please run the cell defining BASE_PATH (e.g., fBQo6uNB_p5H).")
    elif 'mapping_material' not in globals():
        print("Error: mapping_material not found. Please run the cell defining mapping_material (e.g., 6KyyCj6ZACA1).")
    elif 'train_transform' not in globals() or 'validation_transform' not in globals() or 'test_transform' not in globals():
        print("Error: transforms not found. Please run the cell defining image transforms (e.g., GWhPmwnuAKDY).")
    else:
        print("\nProcess: Creating raw datasets...")
        # Ensure raw datasets are globally accessible for _resolve_to_global_indices
        globals()['full_default_raw_dataset'] = WasteDataset(BASE_PATH, mode="default", transform=None)
        globals()['full_real_world_raw_dataset'] = WasteDataset(BASE_PATH, mode="real_world", transform=None)
        print(f"  - Default raw dataset created with {len(globals()['full_default_raw_dataset'])} samples.")
        print(f"  - Real-world raw dataset created with {len(globals()['full_real_world_raw_dataset'])} samples.")

        # Create global_label_list here too for consistency, in case lfw8p9k1AVGN wasn't run or was interrupted
        globals()['global_label_list'] = globals()['full_default_raw_dataset'].labels + globals()['full_real_world_raw_dataset'].labels
        print(f"  - Global label list created with {len(globals()['global_label_list'])} entries (from re-initialization).")

        print("\nProcess: Defining target sizes for train, validation, and test splits...")
        # 2. Define overall target sizes based on 80% train, 10% val, 10% test of total combined data.
        total_combined_data_size = len(globals()['full_default_raw_dataset']) + len(globals()['full_real_world_raw_dataset'])
        target_train_size = int(0.8 * total_combined_data_size)
        target_val_size = int(0.1 * total_combined_data_size)
        target_test_size = total_combined_data_size - target_train_size - target_val_size # Adjust for rounding
        print(f"  - Total combined data size: {total_combined_data_size}")
        print(f"  - Target train size: {target_train_size}")
        print(f"  - Target validation size: {target_val_size}")
        print(f"  - Target test size: {target_test_size}")

        print("\nProcess: Extracting test set (real-world only)...")
        # Ensure target_test_size doesn't exceed available real_world data.
        if target_test_size > len(globals()['full_real_world_raw_dataset']):
            print(f"Warning: Target test size ({target_test_size}) is larger than available real_world data ({len(globals()['full_real_world_raw_dataset'])}). Adjusting test set to use all real_world data.")
            actual_real_world_test_size = len(globals()['full_real_world_raw_dataset'])
        else:
            actual_real_world_test_size = target_test_size

        # 3. Extract Test Set (Real-World Only):
        real_world_remaining_size = len(globals()['full_real_world_raw_dataset']) - actual_real_world_test_size
        real_world_for_train_val_subset, raw_test_subset = random_split(
            globals()['full_real_world_raw_dataset'], [real_world_remaining_size, actual_real_world_test_size]
        )
        globals()['test_dataset'] = TransformedSubset(raw_test_subset, transform=test_transform)
        print(f"  - Test dataset created with {len(globals()['test_dataset'])} samples (real-world only).")
        print(f"  - Remaining real-world samples for train/val pool: {len(real_world_for_train_val_subset)}")

        print("\nProcess: Creating combined pool for training and validation...")
        # 4. Create Combined Pool for Training and Validation:
        train_val_raw_pool = ConcatDataset([globals()['full_default_raw_dataset'], real_world_for_train_val_subset])
        print(f"  - Combined train/val pool created with {len(train_val_raw_pool)} samples.")

        print("\nProcess: Splitting training and validation sets from the combined pool...")
        # 5. Split Training and Validation from the combined pool
        train_val_pool_size = len(train_val_raw_pool)
        adjusted_train_size_from_pool = int((target_train_size / (target_train_size + target_val_size)) * train_val_pool_size)
        adjusted_val_size_from_pool = train_val_pool_size - adjusted_train_size_from_pool

        raw_train_subset, raw_val_subset = random_split(
            train_val_raw_pool, [adjusted_train_size_from_pool, adjusted_val_size_from_pool]
        )
        print(f"  - Raw training subset created with {len(raw_train_subset)} samples.")
        print(f"  - Raw validation subset created with {len(raw_val_subset)} samples.")

        print("\nProcess: Applying transforms and creating final datasets...")
        # 6. Apply Transforms and Create Final Datasets:
        globals()['train_dataset'] = TransformedSubset(raw_train_subset, transform=train_transform)
        globals()['validation_dataset'] = TransformedSubset(raw_val_subset, transform=validation_transform)
        print(f"  - Final train_dataset created with {len(globals()['train_dataset'])} samples and train_transform.")
        print(f"  - Final validation_dataset created with {len(globals()['validation_dataset'])} samples and validation_transform.")

        print("\nProcess: Getting class labels...")
        # 7. Get classes from one of the raw datasets.
        globals()['classes'] = globals()['full_default_raw_dataset'].classes
        print(f"  - Classes defined: {globals()['classes']}")
        print("Datasets and classes re-initialized successfully.")

# Assuming train_dataset, validation_dataset, test_dataset, and classes are now defined.
print("\nProcess: Displaying combined data distribution summary...")
# Use the summary_total_data function to display the combined distribution.
summary_total_data(train_dataset, validation_dataset, globals()['test_dataset'], nama="Persebaran Data Gabungan (Train + Validation + Test)")

print("\nProcess: Displaying individual data distribution summaries...")
# Display individual summaries
summary_dataset(train_dataset, "Training Data")
summary_dataset(validation_dataset, "Validation Data")
summary_dataset(globals()['test_dataset'], "Testing Data")

# New request: display data distribution for 'default' and 'real_world' separately
if 'BASE_PATH' in globals() and 'mapping_material' in globals() and 'WasteDataset' in globals():
    print("\n--- Persebaran Data Terpisah (Raw Data) ---")

    print("Process: Creating raw default dataset for separate summary...")
    # Create raw default dataset
    default_raw_dataset = WasteDataset(BASE_PATH, mode="default", transform=None)
    summary_dataset(default_raw_dataset, "Data Folder Default")

    print("Process: Creating raw real-world dataset for separate summary...")
    # Create raw real_world dataset
    real_world_raw_dataset = WasteDataset(BASE_PATH, mode="real_world", transform=None)
    summary_dataset(real_world_raw_dataset, "Data Folder Real-World")
else:
    print("\nCannot display separate raw data distributions. Ensure BASE_PATH, mapping_material, and WasteDataset are defined.")

In [ ]:
import os
from PIL import Image
import torchvision.transforms as transforms

# Define a sample image path
# Using 'plastic_soda_bottles' from 'default' subfolder as an example
sample_class_folder = "plastic_soda_bottles"
sample_mode = "default"

# Construct the path to find an image
image_folder_path = os.path.join(BASE_PATH, sample_class_folder, sample_mode)

# Ensure the folder exists and get the first image file
if os.path.exists(image_folder_path):
    image_files = [f for f in os.listdir(image_folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    if image_files:
        sample_image_path = os.path.join(image_folder_path, image_files[0])

        # Load the image
        img = Image.open(sample_image_path).convert("RGB")

        print(f"--- Informasi Gambar Sampel ({sample_class_folder}/{sample_mode}) ---")
        print(f"Path: {sample_image_path}")
        print(f"Ukuran Asli (Lebar x Tinggi): {img.size[0]} x {img.size[1]} piksel")

        # Apply the test_transform to see its effect
        # Ensure test_transform is defined from cell fp3vVSIah7FL or GWhPmwnuAKDY
        if 'test_transform' in globals() and isinstance(test_transform, transforms.Compose):
            transformed_img = test_transform(img)
            # Transformed_img is a tensor, its size is (C, H, W)
            print(f"Ukuran Setelah test_transform (Channels x Tinggi x Lebar): {transformed_img.shape}")
        else:
            print("Variabel 'test_transform' tidak ditemukan atau tidak valid.")
    else:
        print(f"Tidak ada file gambar ditemukan di: {image_folder_path}")
else:
    print(f"Folder tidak ditemukan: {image_folder_path}")

In [ ]:
import torch
import gc

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU cache cleared.")

gc.collect()
print("Python garbage collector run.")

#Menunjukkan Sample Dataset

In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
from torch.utils.data import ConcatDataset, Subset, Dataset as BaseDataset

# Helper function to get labels from various dataset types (copied for self-containment)
# This version relies on `WasteDataset` and `TransformedSubset` being defined globally
# and `classes` (list of class names) also being available in the global scope.
def _get_labels_from_dataset(dataset):
    is_wastedataset_defined = 'WasteDataset' in globals() and isinstance(globals()['WasteDataset'], type)
    is_transformedsubset_defined = 'TransformedSubset' in globals() and isinstance(globals()['TransformedSubset'], type)

    if is_wastedataset_defined and isinstance(dataset, globals()['WasteDataset']):
        return dataset.labels
    elif is_transformedsubset_defined and isinstance(dataset, globals()['TransformedSubset']):
        original_source_dataset = dataset.dataset
        if is_wastedataset_defined and isinstance(original_source_dataset, globals()['WasteDataset']):
            return [original_source_dataset.labels[i] for i in dataset.indices]
        elif isinstance(original_source_dataset, torch.utils.data.Subset):
            actual_waste_dataset = original_source_dataset.dataset
            actual_original_indices = original_source_dataset.indices
            if is_wastedataset_defined and isinstance(actual_waste_dataset, globals()['WasteDataset']):
                return [actual_waste_dataset.labels[actual_original_indices[i]] for i in dataset.indices]
            else:
                raise TypeError(f"Unsupported base dataset type within TransformedSubset (nested Subset): {type(actual_waste_dataset)}")
        else:
            raise TypeError(f"Unsupported base dataset type within TransformedSubset: {type(original_source_dataset)}")
    elif isinstance(dataset, ConcatDataset):
        all_labels = []
        for ds in dataset.datasets:
            all_labels.extend(_get_labels_from_dataset(ds))
        return all_labels
    elif isinstance(dataset, Subset):
        if is_wastedataset_defined and isinstance(dataset.dataset, globals()['WasteDataset']):
             return [dataset.dataset.labels[i] for i in dataset.indices]
        else:
             raise TypeError(f"Unsupported base dataset type within torch.utils.data.Subset: {type(dataset.dataset)}")
    else:
        if hasattr(dataset, 'labels'):
            return dataset.labels
        elif 'classes' in globals() and isinstance(globals()['classes'], list):
            inferred_labels = []
            for i in range(len(dataset)):
                _, label_idx = dataset[i]
                if isinstance(label_idx, int) and 0 <= label_idx < len(globals()['classes']):
                    inferred_labels.append(globals()['classes'][label_idx])
                else:
                    inferred_labels.append(str(label_idx))
            return inferred_labels
        else:
            raise TypeError(f"Unsupported dataset type and no global 'classes' for label inference: {type(dataset)}")

def tampilkan_sample_beragam(dataset):
    plt.figure(figsize=(15,3))

    # Get all string labels using the helper function
    all_string_labels = _get_labels_from_dataset(dataset)

    # Get unique materials (classes) from the labels found
    material_unik = sorted(list(set(all_string_labels)))
    index_terpilih = []

    # Find one random index for each unique material from the full list of labels
    for m in material_unik:
        idx_list = [i for i, label_str in enumerate(all_string_labels) if label_str == m]
        if idx_list: # Ensure there are samples for this material
            index_terpilih.append(random.choice(idx_list))
        else:
            print(f"Warning: No samples found for material '{m}' in the provided dataset.")

    # Sort chosen indices to keep the display order consistent with material_unik
    index_terpilih.sort(key=lambda idx: material_unik.index(all_string_labels[idx]))

    for i, idx in enumerate(index_terpilih):
        img_tensor, label_idx = dataset[idx] # dataset[idx] returns img_tensor, label_idx (int)

        # Convert tensor to numpy array and reorder dimensions for matplotlib (C, H, W -> H, W, C)
        img = img_tensor.permute(1,2,0).numpy()

        # Denormalize for correct color display
        # These mean/std values are standard ImageNet stats used in your transforms
        mean = np.array([0.485, 0.456, 0.406])
        std  = np.array([0.229, 0.224, 0.225])
        img = (img * std) + mean
        img = np.clip(img, 0, 1) # Clip values to [0, 1] range for imshow

        # Get the material name using the global 'classes' list and the label_idx (integer)
        if 'classes' in globals() and isinstance(label_idx, int) and 0 <= label_idx < len(globals()['classes']):
            material = globals()['classes'][label_idx]
        else:
            material = f"Unknown (idx: {label_idx})" # Fallback if classes not available or index invalid

        plt.subplot(1, len(index_terpilih), i+1)
        plt.imshow(img)
        plt.title(f"{material}") # Simplified title, removed `folder_asli`
        plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
print("Sample:")
tampilkan_sample_beragam(train_dataset)

### Contoh Gambar Setelah Preprocessing (Tanpa Denormalisasi)

Berikut adalah contoh gambar setelah tahap preprocessing (resize, rotasi, flip, ToTensor, dan Normalize) sebelum masuk ke model. Perhatikan bahwa gambar mungkin terlihat tidak natural karena belum di-denormalisasi untuk tampilan manusia.

In [ ]:
import matplotlib.pyplot as plt
import random
import numpy as np

# Assuming _get_labels_from_dataset is defined in a previous cell (like qoKzoUprLh63)
# and 'classes' list is also globally available.

def tampilkan_sample_preprocessed(dataset, num_samples=5):
    plt.figure(figsize=(15, 3))

    # Get all string labels using the helper function
    all_string_labels = _get_labels_from_dataset(dataset)

    # Ambil beberapa sampel acak dari dataset
    # Ensure we don't ask for more samples than available in the dataset
    actual_num_samples = min(num_samples, len(dataset))
    sample_indices = random.sample(range(len(dataset)), actual_num_samples)

    for i, idx in enumerate(sample_indices):
        img_tensor, label_idx = dataset[idx]

        # Convert tensor to numpy array and reorder dimensions for matplotlib (C, H, W -> H, W, C)
        img_display = img_tensor.permute(1, 2, 0).numpy()

        # Untuk visualisasi, kita bisa menggeser rentang nilai menjadi [0,1] secara kasar.
        # Ini bukan denormalisasi yang sebenarnya, hanya penyesuaian untuk `imshow`.
        # Dikarenakan gambar sudah dinormalisasi dengan mean/std, nilai piksel bisa di luar [0,1].
        # Normalisasi min-max ini membantu `imshow` menampilkannya dengan benar.
        img_display = (img_display - img_display.min()) / (img_display.max() - img_display.min())

        # Get the material name using the global 'classes' list and the label_idx (integer)
        if 'classes' in globals() and isinstance(label_idx, int) and 0 <= label_idx < len(globals()['classes']):
            material = globals()['classes'][label_idx]
        else:
            material = f"Unknown (idx: {label_idx})" # Fallback if classes not available or index invalid

        plt.subplot(1, actual_num_samples, i + 1)
        plt.imshow(img_display)
        plt.title(f"{material}\n(Preprocessed)")
        plt.axis("off")
    plt.show()

print("Sample preprocessed images (raw tensor visualization):")
tampilkan_sample_preprocessed(train_dataset, num_samples=5)

#Visualisasi


In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns

import matplotlib.pyplot as plt
import PIL.Image as Image

base_dir = BASE_PATH  # ganti sesuai lokasi
classes = ["plastic", "paper", "glass", "metal", "organic", "textile"]

In [ ]:
import random

def create_data_summary(base_path, mapping_material, target_classes):
    summary_data = {}
    sampled_image_paths = {}

    # First, find one sample image for each target class using the mapping
    for original_folder, mapped_class in mapping_material.items():
        if mapped_class in target_classes and mapped_class not in sampled_image_paths:
            # Try 'default' subfolder first
            default_path = os.path.join(base_path, original_folder, 'default')
            if os.path.exists(default_path):
                img_files = [f for f in os.listdir(default_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
                if img_files:
                    sampled_image_paths[mapped_class] = os.path.join(default_path, random.choice(img_files))
                    continue

            # If no 'default' or no images, try 'real_world'
            real_world_path = os.path.join(base_path, original_folder, 'real_world')
            if os.path.exists(real_world_path):
                img_files = [f for f in os.listdir(real_world_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
                if img_files:
                    sampled_image_paths[mapped_class] = os.path.join(real_world_path, random.choice(img_files))
                    continue

    # Now, process the sampled images to create the summary
    for c in target_classes:
        if c in sampled_image_paths:
            img_path = sampled_image_paths[c]
            img = Image.open(img_path).convert('RGB')
            img = img.resize((100, 100)) # Keep original resize
            img_array = np.array(img)

            # Extract RGB channel
            summary_data[f"{c}_Red"] = img_array[:, :, 0].flatten()
            summary_data[f"{c}_Green"] = img_array[:, :, 1].flatten()
            summary_data[f"{c}_Blue"] = img_array[:, :, 2].flatten()
        else:
            print(f"Warning: No sample image found for class '{c}'")

    if summary_data:
        # Return the DataFrame of raw pixel data, not its description
        return pd.DataFrame(summary_data)
    else:
        print("Error: No data to summarize.")
        return pd.DataFrame() # Return empty DataFrame or handle as appropriate


In [ ]:
# Call the modified function to get the raw pixel data DataFrame
raw_pixel_data_df = create_data_summary(base_dir, mapping_material, classes)

# Prepare to collect descriptive statistics
stats_list = []

# Iterate through each class and each channel to get descriptive statistics
for cls in classes:
    for channel in ['Red', 'Green', 'Blue']:
        col_name = f'{cls}_{channel}'
        if col_name in raw_pixel_data_df.columns:
            # Get descriptive statistics for the current column
            desc_stats = raw_pixel_data_df[col_name].describe()
            stats_list.append({
                'Class': cls,
                'Channel': channel,
                'Mean': desc_stats['mean'],
                'Std': desc_stats['std'],
                'Min': desc_stats['min'],
                '25%': desc_stats['25%'],
                '50%': desc_stats['50%'],
                '75%': desc_stats['75%'],
                'Max': desc_stats['max']
            })

# Create a DataFrame from the collected statistics
descriptive_stats_df = pd.DataFrame(stats_list)

print("\nDescriptive Statistics per Class and RGB Channel (Original Format):")
display(descriptive_stats_df)

print("\nDescriptive Statistics per Class and RGB Channel (Transposed Format):")
# Transpose the DataFrame to change its display position
transposed_stats_df = descriptive_stats_df.set_index(['Class', 'Channel']).T
display(transposed_stats_df)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Reconstruct df_vis from descriptive_stats_df to get mean RGB per class
# This assumes descriptive_stats_df has been created in a previous cell (e.g., jEIv7XZsZSNK)
mean_rgb_per_class = {
    'Class': descriptive_stats_df['Class'].unique(),
    'Mean_Red': descriptive_stats_df[descriptive_stats_df['Channel'] == 'Red']['Mean'].values,
    'Mean_Green': descriptive_stats_df[descriptive_stats_df['Channel'] == 'Green']['Mean'].values,
    'Mean_Blue': descriptive_stats_df[descriptive_stats_df['Channel'] == 'Blue']['Mean'].values
}
df_vis = pd.DataFrame(mean_rgb_per_class).set_index('Class')

# Calculate the correlation matrix for mean RGB intensities
correlation_matrix_mean_rgb = df_vis.corr()

print("\nMatriks Korelasi (Pearson) untuk Rata-rata Intensitas RGB:")
display(correlation_matrix_mean_rgb)

# Visualize the correlation matrix using a heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix_mean_rgb, annot=True, cmap='coolwarm', fmt=".3f")
plt.title('Heatmap Matriks Korelasi Rata-rata Intensitas RGB')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from IPython.display import display, Markdown

# Scatter plots based on the data that generated the correlation matrix (df_vis)
print("Visualisasi Scatter Plot dari Rata-rata Intensitas RGB:")

# Use sns.pairplot for a quick overview of all pairwise relationships
sns.pairplot(df_vis, diag_kind='kde')
plt.suptitle('Pairwise Scatter Plot of Mean RGB Intensities', y=1.02)
plt.show()

# Individual scatter plots for clearer titles/details if needed
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
sns.scatterplot(data=df_vis, x='Mean_Red', y='Mean_Green')
plt.title('Mean Red vs Mean Green')
plt.xlabel('Rata-rata Intensitas Merah')
plt.ylabel('Rata-rata Intensitas Hijau')

plt.subplot(1, 3, 2)
sns.scatterplot(data=df_vis, x='Mean_Red', y='Mean_Blue')
plt.title('Mean Red vs Mean Blue')
plt.xlabel('Rata-rata Intensitas Merah')
plt.ylabel('Rata-rata Intensitas Biru')

plt.subplot(1, 3, 3)
sns.scatterplot(data=df_vis, x='Mean_Green', y='Mean_Blue')
plt.title('Mean Green vs Mean Blue')
plt.xlabel('Rata-rata Intensitas Hijau')
plt.ylabel('Rata-rata Intensitas Biru')

plt.tight_layout()
plt.show()

display(Markdown("## Analisis Data Eksplorasi (EDA) dan Scatter Plot\n\n**EDA (Exploratory Data Analysis)** adalah pendekatan untuk menganalisis set data untuk meringkas karakteristik utamanya, seringkali dengan metode visual. Ini adalah langkah penting dalam proses analisis data untuk menemukan pola, mendeteksi anomali, menguji hipotesis, dan memeriksa asumsi dengan bantuan statistik ringkasan dan representasi grafis.\n\n### Keterkaitan dengan Scatter Plot dan Matriks Korelasi:\n\n*   **Matriks Korelasi** yang telah dibuat sebelumnya memberikan gambaran kuantitatif tentang hubungan linear antar variabel. Nilai korelasi mendekati 1 menunjukkan hubungan positif yang kuat, mendekati -1 menunjukkan hubungan negatif yang kuat, dan mendekati 0 menunjukkan tidak ada hubungan linear.\n*   **Scatter Plot** adalah visualisasi grafis yang sangat berguna dalam EDA untuk memperlihatkan hubungan antara dua variabel kuantitatif. Dari scatter plot di atas, kita dapat secara visual mengkonfirmasi apa yang ditunjukkan oleh matriks korelasi:\n    *   **Pola**: Jika titik-titik data cenderung membentuk garis lurus yang naik, itu menunjukkan korelasi positif. Jika cenderung membentuk garis lurus yang turun, itu korelasi negatif. Sebaran titik yang acak menunjukkan korelasi yang rendah atau tidak ada.\n    *   **Kekuatan Hubungan**: Semakin rapat titik-titik data mengumpul di sekitar garis imajiner, semakin kuat korelasinya.\n    *   **Pencilan (Outliers)**: Scatter plot juga efektif untuk mengidentifikasi pencilan, yaitu titik data yang jauh dari pola umum. Pencilan dapat mempengaruhi nilai korelasi secara signifikan.\n\nDalam konteks rata-rata intensitas RGB, scatter plot ini menunjukkan bagaimana intensitas satu saluran warna berubah seiring dengan perubahan intensitas saluran warna lainnya dalam sampel gambar yang diambil. Hubungan positif yang kuat antar saluran, seperti yang terlihat pada matriks korelasi dan scatter plot, adalah hal yang umum karena perubahan cahaya atau warna pada objek biasanya mempengaruhi ketiga saluran RGB secara bersamaan."))

In [ ]:
from collections import Counter
from scipy import stats
import numpy as np
import pandas as pd

# Combine labels from training and testing datasets
all_labels = train_dataset.labels + test_dataset.labels
total_class_counts = Counter(all_labels)

# Create a DataFrame for total distribution
df_stats = pd.DataFrame(list(total_class_counts.items()), columns=['Class', 'Count'])

print("Distribusi jumlah gambar per kelas (Total):")
print(df_stats.to_string(index=False))
print("-" * 50)

image_counts = df_stats['Count'].values

mean_val = np.mean(image_counts)
median_val = np.median(image_counts)

# stats.mode returns a ModeResult object, access the mode value
mode_result = stats.mode(image_counts, keepdims=True)
mode_val = mode_result.mode[0]

variance_val = np.var(image_counts, ddof=1) # ddof=1 for sample variance
std_dev_val = np.std(image_counts, ddof=1) # ddof=1 for sample standard deviation

print("Statistik Deskriptif Jumlah Gambar:")
print(f"Mean            : {mean_val:.2f}")
print(f"Median          : {median_val:.2f}")
print(f"Mode            : {mode_val}")
print(f"Varians         : {variance_val:.2f}")
print(f"Standar Deviasi : {std_dev_val:.2f}")

In [ ]:
### Analisis Korelasi Rata-rata Intensitas Piksel RGB

### Dari matriks korelasi dan visualisasi heatmap, terlihat bahwa terdapat korelasi positif yang kuat antara rata-rata intensitas piksel Red, Green, dan Blue pada batch gambar yang diambil. Hal ini wajar karena ketiga channel warna ini seringkali saling bergantung dan berubah secara proporsional dalam gambar yang sama, terutama setelah proses normalisasi. Scatter plot juga menunjukkan tren hubungan positif, di mana peningkatan intensitas pada satu channel cenderung diikuti oleh peningkatan pada channel lainnya, meskipun dengan derajat kekuatan yang bervariasi antara pasangan channel.

### Analisis Korelasi Rata-rata Intensitas Piksel RGB

Dari matriks korelasi dan visualisasi heatmap, terlihat bahwa terdapat korelasi positif yang kuat antara rata-rata intensitas piksel Red, Green, dan Blue pada batch gambar yang diambil. Hal ini wajar karena ketiga channel warna ini seringkali saling bergantung dan berubah secara proporsional dalam gambar yang sama, terutama setelah proses normalisasi. Scatter plot juga menunjukkan tren hubungan positif, di mana peningkatan intensitas pada satu channel cenderung diikuti oleh peningkatan pada channel lainnya, meskipun dengan derajat kekuatan yang bervariasi antara pasangan channel.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# 1. Histogram untuk satu kelas target (misalnya 'plastic')
target_class = 'plastic' # Anda bisa mengubah kelas target di sini

plt.figure(figsize=(8, 5))

sns.histplot(raw_pixel_data_df[f'{target_class}_Red'], color="red", label="Red", kde=True, binrange=(0, 255), bins=50)
sns.histplot(raw_pixel_data_df[f'{target_class}_Green'], color="green", label="Green", kde=True, binrange=(0, 255), bins=50)
sns.histplot(raw_pixel_data_df[f'{target_class}_Blue'], color="blue", label="Blue", kde=True, binrange=(0, 255), bins=50)

plt.xlim(0, 255)
plt.title(f'Distribusi Intensitas Warna RGB: Kelas {target_class.capitalize()}')
plt.xlabel('Nilai Piksel (0-255)')
plt.ylabel('Frekuensi')
plt.legend()
plt.show()

# 2. Distribusi intensitas warna Merah untuk beberapa kelas
sample_classes = classes[:3] # Mengambil 3 kelas pertama, bisa disesuaikan
cols_to_use = [f'{c}_Red' for c in sample_classes]

# Memastikan kolom-kolom yang akan digunakan ada di raw_pixel_data_df
# Beberapa kolom mungkin belum ada jika `raw_pixel_data_df` hanya berisi sampel
# Jadi, kita hanya memilih kolom yang benar-benar ada
existing_cols = [col for col in cols_to_use if col in raw_pixel_data_df.columns]

if existing_cols:
    subset_data = raw_pixel_data_df[existing_cols]

    melted_data = subset_data.melt(var_name='Class', value_name='Red_Intensity')
    # Memperbaiki nama kelas dalam 'Class' dari 'plastic_Red' menjadi 'plastic'
    melted_data['Class'] = melted_data['Class'].apply(lambda x: x.split('_')[0])

    # Define a custom title formatter function
    def format_col_title(data):
        return f"Distribusi Intensitas Merah: Kelas {data['col_name'].capitalize()}"

    g = sns.displot(data=melted_data, x="Red_Intensity", col="Class", kde=True, color="Red", facet_kws=dict(sharey=False))
    g.set(xlim=(0, 255))
    g.set_axis_labels('Nilai Piksel Merah (0-255)', 'Frekuensi')
    g.set_titles(col_template=format_col_title)
    plt.suptitle('Distribusi Intensitas Warna Merah untuk Beberapa Kelas', y=1.02) # y=1.02 untuk judul di atas semua subplots
    plt.tight_layout()
    plt.show()
else:
    print("Tidak ada data kolom Merah yang ditemukan untuk kelas sampel yang dipilih.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# 2. Box Plot
# Mengambil satu batch dari train_loader
# Perhatikan: `images` adalah batch tensor (B, C, H, W) dan sudah dinormalisasi.
# Nilai piksel setelah normalisasi bisa berada di luar rentang [0,1].
images, labels = next(iter(train_loader))

# Mengonversi ke numpy dan menghitung rata-rata per channel
# img[0,:,:] adalah Red, img[1,:,:] adalah Green, img[2,:,:] adalah Blue
r_means = [img[0, :, :].cpu().numpy().mean() for img in images]
g_means = [img[1, :, :].cpu().numpy().mean() for img in images]
b_means = [img[2, :, :].cpu().numpy().mean() for img in images]

df_vis = pd.DataFrame({
    'Mean_Red': r_means,
    'Mean_Green': g_means,
    'Mean_Blue': b_means
})

plt.figure(figsize=(10, 5))
sns.set_theme(style="whitegrid")

# Melt the df_vis for easier plotting with seaborn.boxplot to get separate box plots per channel
df_vis_melted = df_vis.melt(var_name='Channel', value_name='Mean_Intensity')

sns.boxplot(data=df_vis_melted, y='Channel', x='Mean_Intensity', orient="h", palette=["#ff6666", "#66ff66", "#6666ff"])

plt.title('Box Plot: Distribusi Intensitas RGB Rata-rata (Batch Training)', fontsize=14, pad=15)
plt.xlabel('Intensitas Piksel Rata-rata (Setelah Normalisasi)', fontsize=12)
# plt.xlim(0, 1) # Dikomentari karena nilai bisa di luar [0,1] setelah normalisasi
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


# Catplot

# 1. Tentukan kategori yang ingin dibandingkan
# Menggunakan kelas yang valid dari `classes` yang ada
# Contoh user menggunakan ['paper', 'metal', 'white-glass', 'plastic'],
# 'white-glass' tidak ada di `classes`, jadi kita ganti dengan 'glass'.
valid_sample_classes = ['paper', 'metal', 'glass', 'plastic']

# 2. Filter data dari ds_table (Hasil EDA Statistik sebelumnya)
# PENTING: `ds_table` berisi statistik deskriptif (count, mean, std, min, max, 25%, 50%, 75%)
# bukan nilai piksel mentah. Catplot ini akan memvisualisasikan distribusi dari statistik tersebut.
cols_to_use_catplot = [f'{c}_Red' for c in valid_sample_classes if f'{c}_Red' in raw_pixel_data_df.columns]
subset_df_catplot = raw_pixel_data_df[cols_to_use_catplot]

# 3. Transformasi data (Melt) untuk kebutuhan Catplot
df_melted_catplot = subset_df_catplot.melt(var_name='Category', value_name='Intensity')
df_melted_catplot['Category'] = df_melted_catplot['Category'].apply(lambda x: x.replace('_Red', ''))

# 4. Visualisasi Catplot (Box Plot antar Kategori)
g = sns.catplot(
    data=df_melted_catplot,
    x="Category",
    y="Intensity",
    kind="box",
    hue="Category",
    legend=False,
    palette="husl",
    height=6,
    aspect=1.5
)

# 5. Merapikan label dan skala
g.set_xticklabels(rotation=45)
# PENTING: Skala 0-255 akan sangat mengkompresi boxplot karena kolom 'Intensity'
# di sini adalah nilai-nilai statistik (seperti count=10000, mean, std, dll.),
# bukan distribusi piksel individu. Beberapa statistik seperti 'count' akan terpotong (clipped).
# Jika tujuannya membandingkan statistik yang berbeda, mungkin rentang ini perlu disesuaikan.
g.set(ylim=(0, 255))
plt.title('EDA: Komparasi Sebaran Statistik Intensitas Red antar Kategori Sampah dari ds_table', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 7))
sns.jointplot(data=df_vis, x='Mean_Red', y='Mean_Green', kind='reg', height=7, scatter_kws={'alpha':0.6}, line_kws={'color':'red'})
plt.suptitle('Joint Plot: Korelasi Rata-rata Intensitas Merah dan Hijau', y=1.02) # y for title position
plt.xlabel('Rata-rata Intensitas Merah')
plt.ylabel('Rata-rata Intensitas Hijau')
plt.show()

#BALANCING



#Model: ResNet50

###Load Model

In [ ]:
import torch
import torch.nn as nn
import timm

# Define device (GPU if available, else CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Menggunakan perangkat: {device}")

# Load the ResNet50 model with pre-trained ImageNet weights from timm
model = timm.create_model('resnet50', pretrained=True)

# Modify the final classification layer to match the number of classes
# 'classes' variable is assumed to be available from previous cells (e.g., ['glass', 'metal', 'organic', 'paper', 'plastic', 'textile'])
num_classes = len(classes)
model.fc = nn.Linear(model.fc.in_features, num_classes)

# Move the model to the appropriate device (CPU or GPU)
model = model.to(device)

print("PyTorch ResNet50 model loaded and configured.")

In [ ]:
import pandas as pd
import timm
import torch.nn as nn

# Re-create the ResNet50 model specifically for parameter inspection
# This ensures we inspect ResNet50 parameters without affecting the current 'model' (which is EfficientNet).
resnet_model_for_inspection = timm.create_model('resnet50', pretrained=True)
# Adjust the final layer to match the number of classes, similar to the original ResNet setup.
resnet_model_for_inspection.fc = nn.Linear(resnet_model_for_inspection.fc.in_features, len(classes))

# Prepare data for the table
parameters_data = []
for name, param in resnet_model_for_inspection.named_parameters():
    # Only include trainable parameters
    if param.requires_grad:
        parameters_data.append({
            "Nama Parameter": name,
            "Bentuk (Shape)": str(tuple(param.shape)), # Convert torch.Size to tuple for better display
            "Jumlah Elemen Total": param.numel(),
            "Membutuhkan Gradien": param.requires_grad
        })

# Create a DataFrame
df_parameters = pd.DataFrame(parameters_data)

print("Parameter Model ResNet50:")
display(df_parameters)

print("\nJumlah total parameter yang dapat dilatih (trainable parameters) di ResNet50:")
print(sum(p.numel() for p in resnet_model_for_inspection.parameters() if p.requires_grad))

##Train Model

In [ ]:
import torch.cuda.amp as amp
import torch.nn as nn
import torch.optim as optim
import collections
import os

# Assuming 'model' and 'device' are already defined in cell 'qLMKCjqOAc95'
# The model is a PyTorch ResNet50 and moved to the correct device.

print("Starting new training run using the PyTorch ResNet50 model from a previous cell.")
print(f"Training on device: {device}")

# Helper function to get labels from various dataset types (copied for self-containment)
def _get_labels_from_dataset(dataset):
    if isinstance(dataset, WasteDataset):
        return dataset.labels
    elif isinstance(dataset, TransformedSubset):
        original_source_dataset = dataset.dataset
        if isinstance(original_source_dataset, WasteDataset):
            return [original_source_dataset.labels[i] for i in dataset.indices]
        elif isinstance(original_source_dataset, torch.utils.data.Subset):
            actual_waste_dataset = original_source_dataset.dataset
            actual_original_indices = original_source_dataset.indices
            return [actual_waste_dataset.labels[actual_original_indices[i]] for i in dataset.indices]
        else:
            raise TypeError(f"Unsupported base dataset type within TransformedSubset: {type(original_source_dataset)}")
    elif isinstance(dataset, torch.utils.data.ConcatDataset):
        all_labels = []
        for ds in dataset.datasets:
            all_labels.extend(_get_labels_from_dataset(ds))
        return all_labels
    elif isinstance(dataset, torch.utils.data.Subset):
        return [dataset.dataset.labels[i] for i in dataset.indices]
    else:
        if hasattr(dataset, 'labels'):
            return dataset.labels
        else:
            print("Warning: Attempting to infer labels from __getitem__ for an unsupported dataset type.")
            inferred_labels = []
            for i in range(len(dataset)):
                _, label = dataset[i]
                if isinstance(label, int):
                    inferred_labels.append(classes[label])
                else:
                    inferred_labels.append(label)
            return inferred_labels


# Get all class labels from the training dataset (`train_dataset` is already defined in previous cells)
train_labels_str = _get_labels_from_dataset(train_dataset)

# Map string labels to integer indices using `class_to_idx` from the original dataset
class_to_idx = {c: i for i, c in enumerate(classes)}
train_labels_idx = [class_to_idx[label] for label in train_labels_str]

# Count occurrences of each class
class_counts = collections.Counter(train_labels_idx)
total_samples = sum(class_counts.values())

# Calculate inverse frequency weights
weights = [0] * len(classes)
for i, c in enumerate(classes):
    idx = class_to_idx[c]
    if class_counts[idx] > 0:
        weights[idx] = total_samples / (len(classes) * class_counts[idx])
    else:
        weights[idx] = 0.0

class_weights = torch.tensor(weights, dtype=torch.float).to(device)
print(f"Calculated class weights: {class_weights}")

# Define criterion and optimizer
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# Initialize GradScaler for Automatic Mixed Precision (AMP)
# Use the updated torch.amp.autocast for future compatibility if available
if hasattr(torch.amp, 'autocast'):
    scaler = torch.amp.GradScaler() if device.type == 'cuda' else None
else:
    # Fallback for older PyTorch versions or if torch.amp is not available
    scaler = amp.GradScaler() if device.type == 'cuda' else None
    if scaler:
        print("Using deprecated `torch.cuda.amp.GradScaler` but will try to use `torch.amp.autocast`.")

print(f"Memulai pelatihan pada {device}")

# Training Loop with 50 Epochs
num_epochs = 10 # Set to 10 epochs as requested
print(f"Training for {num_epochs} epochs...")

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for batch_idx, (imgs, labels) in enumerate(train_loader):
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()

        # Use autocast for mixed precision if scaler is initialized
        if scaler and device.type == 'cuda':
            if hasattr(torch.amp, 'autocast'):
                with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                    outputs = model(imgs)
                    loss = criterion(outputs, labels)
            else:
                # Fallback to old autocast if new one not available
                with amp.autocast(enabled=True):
                    outputs = model(imgs)
                    loss = criterion(outputs, labels)
        else:
            # Standard float32 training for CPU or when AMP is not used
            outputs = model(imgs)
            loss = criterion(outputs, labels)

        if scaler:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        total_loss += loss.item()

        # Print progress every 10 batches (changed from 100)
        if (batch_idx + 1) % 10 == 0 or (batch_idx + 1) == len(train_loader):
            print(f"  Epoch {epoch+1}/{num_epochs}, Batch {batch_idx+1}/{len(train_loader)}, Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs} selesai, Avg Loss: {avg_loss:.4f}\n")

    if device.type == 'cuda':
        torch.cuda.empty_cache()

# Save the trained model to a new file
model_save_path = "resnet50_trained_model_50epochs.pth" # Unique save path
torch.save(model.state_dict(), model_save_path)
print(f"\nModel telah disimpan ke: {model_save_path}")
print("Catatan: Model PyTorch disimpan dalam format .pth.")

#TRAIN WITH EARLY STOPPIN

In [ ]:
import torch.cuda.amp as amp
import torch.nn as nn
import torch.optim as optim
import collections
import os
import copy # To save best model state_dict
from torch.utils.data import DataLoader, ConcatDataset, Subset

# Assume WasteDataset and TransformedSubset are available from previous cells (kvSfoF-eAQfh and lfw8p9k1AVGN)
# Assume train_dataset, validation_dataset, classes, device, train_transform, test_transform are globally available from previous cells.

# Helper function to recursively find the base WasteDataset and its original indices
# relative to the full combined raw dataset (default + real_world).
def _resolve_to_global_indices(dataset):
    # Base case: WasteDataset
    if isinstance(dataset, globals().get('WasteDataset')):
        if 'full_default_raw_dataset' in globals() and dataset is globals()['full_default_raw_dataset']:
            return list(range(len(dataset))) # Indices for default dataset
        elif 'full_real_world_raw_dataset' in globals() and dataset is globals()['full_real_world_raw_dataset']:
            # Offset needed because global_label_list is default labels then real_world labels
            offset = len(globals()['full_default_raw_dataset']) if 'full_default_raw_dataset' in globals() else 0
            return [i + offset for i in range(len(dataset))]
        else:
            # This case should ideally not happen if only the two global raw datasets are used.
            # If WasteDataset was instantiated directly from within this cell, it might not be identified.
            # Fallback to assuming its indices are global within itself, which might be incorrect for a subset.
            print(f"Warning: Unidentified WasteDataset instance encountered: {dataset}. Proceeding with assumption.")
            return list(range(len(dataset)))

    # Recursive cases: TransformedSubset, Subset, ConcatDataset
    elif isinstance(dataset, globals().get('TransformedSubset')):
        # TransformedSubset.indices are relative to its internal .dataset
        resolved_base_indices = _resolve_to_global_indices(dataset.dataset)
        return [resolved_base_indices[i] for i in dataset.indices]

    elif isinstance(dataset, torch.utils.data.Subset):
        # Subset.indices are relative to its internal .dataset
        resolved_base_indices = _resolve_to_global_indices(dataset.dataset)
        return [resolved_base_indices[i] for i in dataset.indices]

    elif isinstance(dataset, ConcatDataset):
        all_global_indices = []
        for ds in dataset.datasets:
            all_global_indices.extend(_resolve_to_global_indices(ds))
        return all_global_indices

    else:
        raise TypeError(f"Unsupported dataset type for resolving global indices: {type(dataset)}")

# Original slow fallback method (kept for robustness, though should ideally be avoided)
def _get_labels_from_dataset_slow(dataset):
    if hasattr(dataset, 'labels'):
        return dataset.labels
    elif 'classes' in globals() and isinstance(globals()['classes'], list):
        inferred_labels = []
        try:
            for i in range(len(dataset)):
                _, label_idx = dataset[i]
                if isinstance(label_idx, int) and 0 <= label_idx < len(globals()['classes']):
                    inferred_labels.append(globals()['classes'][label_idx])
                else:
                    inferred_labels.append(str(label_idx))
            return inferred_labels
        except Exception as e:
            print(f"Warning: Could not infer labels from dataset __getitem__ (slow path): {e}")
            return []
    else:
        print(f"Warning: Unsupported dataset type and no global 'classes' for label inference (slow path): {type(dataset)}")
        return []

# Modified _get_labels_from_dataset to leverage global_label_list
def _get_labels_from_dataset(dataset):
    if 'global_label_list' not in globals():
        print("Error: 'global_label_list' not found. Please ensure cell 'lfw8p9k1AVGN' has been executed or datasets re-initialized.")
        # Fallback to slow method if global_label_list isn't there
        return _get_labels_from_dataset_slow(dataset)

    # Direct return for WasteDataset if it's the base raw one
    if isinstance(dataset, globals().get('WasteDataset')):
        return dataset.labels

    try:
        # Attempt to get global indices and retrieve labels from global_label_list
        global_indices = _resolve_to_global_indices(dataset)
        return [globals()['global_label_list'][idx] for idx in global_indices]
    except (TypeError, ValueError) as e:
        print(f"Warning: Could not resolve global indices for {type(dataset)} ({e}). Falling back to slow label inference.")
        return _get_labels_from_dataset_slow(dataset)

# --- 2. Create DataLoaders from existing datasets ---
# Use the existing train_dataset and validation_dataset created in cell lfw8p9k1AVGN.
# The batch_size is also available globally from lfw8p9k1AVGN (value: 256).
print("Initializing DataLoaders...")
# Note: train_dataset already has train_transform applied, and validation_dataset has validation_transform (test_transform) applied.
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(validation_dataset, batch_size=batch_size, shuffle=False, num_workers=0) # No shuffle for validation

print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(validation_dataset)}")
print(f"Batch size for training and validation: {batch_size}")
print("DataLoaders created.")
print("-" * 50)

# --- 3. Calculate class weights based on the training dataset ---
print("Extracting labels from training dataset for class weight calculation...")
# Get all class labels from the training dataset (globally available `train_dataset`)
train_labels_str = _get_labels_from_dataset(train_dataset)
print(f"Extracted {len(train_labels_str)} labels.")

print("Calculating inverse frequency class weights...")
# Map string labels to integer indices using `class_to_idx`
# `classes` variable is assumed to be available globally.
class_to_idx = {c: i for i, c in enumerate(classes)}
train_labels_idx = [class_to_idx[label] for label in train_labels_str]

# Count occurrences of each class
class_counts = collections.Counter(train_labels_idx)
total_samples = sum(class_counts.values())

# Calculate inverse frequency weights
weights = [0] * len(classes)
for i, c in enumerate(classes):
    idx = class_to_idx[c]
    if class_counts[idx] > 0:
        weights[idx] = total_samples / (len(classes) * class_counts[idx])
    else:
        weights[idx] = 0.0 # Handle classes with no samples (though unlikely for main training set)

class_weights = torch.tensor(weights, dtype=torch.float).to(device)
print(f"Calculated class weights for training set: {class_weights}")
print("Class weights calculation complete.")
print("-" * 50)

# --- 4. Setup Optimizer, Loss, & Tracking Variables ---
# `model` (ResNet50 or EfficientNet) and `device` are assumed to be available from previous cells.
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=0.0001) # Learning rate as used previously

# Initialize GradScaler for Automatic Mixed Precision (AMP)
if hasattr(torch.amp, 'autocast'):
    scaler = torch.amp.GradScaler() if device.type == 'cuda' else None
else:
    scaler = amp.GradScaler() if device.type == 'cuda' else None
    if scaler:
        print("Using deprecated `torch.cuda.amp.GradScaler` but will try to use `torch.amp.autocast`.")

# Initialize variables for Early Stopping & Plotting (preparation for the actual training loop)
num_epochs = 15
patience = 5
best_val_loss = float('inf')
patience_counter = 0
best_epoch = -1
best_model_weights = None

train_losses, train_accuracies = [], []
val_losses, val_accuracies = [], []

print(f"Persiapan selesai. Siap memulai pelatihan pada {device}.")
print("Training loop will be executed in a subsequent cell.")
print("-" * 50)

In [ ]:
from tqdm.auto import tqdm

print(f"Memulai pelatihan pada {device}")
print(f"Training for up to {num_epochs} epochs with patience of {patience}...")

for epoch in range(num_epochs):
    # --- Training Phase ---
    model.train()
    current_train_loss = 0
    correct_train_predictions = 0
    total_train_samples = 0

    # Bungkus loader dengan tqdm untuk melihat progress per batch
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]", unit="batch", leave=False)

    for batch_idx, (imgs, labels) in enumerate(train_bar):
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()

        if scaler and device.type == 'cuda':
            if hasattr(torch.amp, 'autocast'):
                with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                    outputs = model(imgs)
                    loss = criterion(outputs, labels)
            else:
                with amp.autocast(enabled=True):
                    outputs = model(imgs)
                    loss = criterion(outputs, labels)
        else:
            outputs = model(imgs)
            loss = criterion(outputs, labels)

        if scaler:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        current_train_loss += loss.item() * imgs.size(0) # Accumulate loss weighted by batch size
        _, predicted = torch.max(outputs.data, 1)
        total_train_samples += labels.size(0)
        correct_train_predictions += (predicted == labels).sum().item()

        # Tambahkan postfix agar metrik loss tampil secara real-time di bar
        train_bar.set_postfix(loss=loss.item())

    avg_train_loss = current_train_loss / total_train_samples
    train_accuracy = correct_train_predictions / total_train_samples
    train_losses.append(avg_train_loss)
    train_accuracies.append(train_accuracy)

    # --- Validation Phase ---
    model.eval()
    current_val_loss = 0
    correct_val_predictions = 0
    total_val_samples = 0

    # Bungkus loader dengan tqdm untuk melihat progress per batch
    val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]", unit="batch", leave=False)

    with torch.no_grad():
        for imgs, labels in val_bar:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)

            current_val_loss += loss.item() * imgs.size(0) # Accumulate loss weighted by batch size
            _, predicted = torch.max(outputs.data, 1)
            total_val_samples += labels.size(0)
            correct_val_predictions += (predicted == labels).sum().item()
            val_bar.set_postfix(loss=loss.item())

    avg_val_loss = current_val_loss / total_val_samples
    val_accuracy = correct_val_predictions / total_val_samples
    val_losses.append(avg_val_loss)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs} selesai.")
    print(f"  Train Loss: {avg_train_loss:.4f}, Train Acc: {train_accuracy:.4f}")
    print(f"  Val Loss: {avg_val_loss:.4f}, Val Acc: {val_accuracy:.4f}\n")

    # --- Early Stopping Logic ---
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_epoch = epoch + 1
        best_model_weights = copy.deepcopy(model.state_dict()) # Save a deep copy of model weights
        patience_counter = 0
        print(f"  Validation loss improved. Saving model weights. Best Val Loss: {best_val_loss:.4f}")
    else:
        patience_counter += 1
        print(f"  Validation loss did not improve. Patience: {patience_counter}/{patience}")
        if patience_counter >= patience:
            print(f"  Early stopping triggered after {epoch+1} epochs. Restoring best model from Epoch {best_epoch}.")
            if best_model_weights is not None:
                model.load_state_dict(best_model_weights) # Load the best model weights
            else:
                print("  Warning: best_model_weights was None, no state_dict to restore.")
            break

    if device.type == 'cuda':
        torch.cuda.empty_cache()

# --- 5. Save the best model and print summary ---
model_save_path = "resnet50_best_early_stop_model.pth"
if best_model_weights is not None:
    torch.save(best_model_weights, model_save_path)
    print(f"\nModel terbaik dari Epoch {best_epoch} (dengan Val Loss: {best_val_loss:.4f}) telah disimpan ke: {model_save_path}")
else:
    print("\nTidak ada model yang disimpan karena pelatihan tidak menghasilkan peningkatan validasi yang signifikan atau diinterupsi.")

print("Catatan: Model PyTorch disimpan dalam format .pth.")

# Make sure the metric lists are globally accessible for future plotting
# These variables are now updated in the current kernel state.
# They will be used to plot the curves in a separate cell, as requested.
print("\nVariabel metrik untuk kurva training vs validation telah disimpan di memori:")
print(f"  train_losses (jumlah epoch): {len(train_losses)}")
print(f"  train_accuracies (jumlah epoch): {len(train_accuracies)}")
print(f"  val_losses (jumlah epoch): {len(val_losses)}")
print(f"  val_accuracies (jumlah epoch): {len(val_accuracies)}")
print(f"  best_epoch (epoch dengan val_loss terbaik): {best_epoch}")

# Optional: Clear some memory after training
# del new_train_dataset, validation_dataset, train_loader, val_loader # These are already globally set by previous cell
if device.type == 'cuda':
    torch.cuda.empty_cache()
import gc
gc.collect()

In [ ]:
import shutil
import os
from google.colab import drive

# Mount Google Drive if not already mounted
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Define the source file path (the .pth file generated in the previous step)
source_file = 'resnet50_best_early_stop_model.pth'

# Define the destination folder in your Google Drive
# You can change 'MyDrive/Colab Notebooks' to your preferred folder path in Drive
drive_destination_folder = '/content/drive/MyDrive/Model_Checkpoints'

# Create the destination folder in Drive if it doesn't exist
os.makedirs(drive_destination_folder, exist_ok=True)

# Define the full destination path for the file in Drive
destination_file = os.path.join(drive_destination_folder, os.path.basename(source_file))

# Copy the file to Google Drive
try:
    shutil.copy(source_file, destination_file)
    print(f"File '{source_file}' berhasil disimpan ke Google Drive di: {destination_file}")
except Exception as e:
    print(f"Gagal menyimpan file ke Google Drive: {e}")


### Bagaimana Cara Mengunduh File dari Google Drive:

Setelah kode di atas dieksekusi, file `.pth` Anda akan tersimpan di Google Drive. Berikut langkah-langkah untuk mengunduhnya:

1.  **Buka Google Drive**: Pergi ke [Google Drive Anda](https://drive.google.com/).
2.  **Navigasi ke Folder**: Cari folder `Model_Checkpoints` (atau nama folder lain yang Anda tentukan di kode di atas) di dalam `My Drive`.
3.  **Temukan File**: Di dalam folder tersebut, Anda akan menemukan file `resnet50_best_early_stop_model.pth`.
4.  **Unduh**: Klik kanan pada file tersebut, lalu pilih `Unduh`.

In [ ]:
import pandas as pd
import shutil
import os
from google.colab import drive

# 1. Create a DataFrame from the collected metrics
metrics_df = pd.DataFrame({
    'epoch': range(1, len(train_losses) + 1),
    'train_loss': train_losses,
    'val_loss': val_losses,
    'train_accuracy': train_accuracies,
    'val_accuracy': val_accuracies
})

# 2. Define the CSV file path
csv_filename = 'training_metrics.csv'

# 3. Save the DataFrame to a CSV file
metrics_df.to_csv(csv_filename, index=False)
print(f"Training metrics saved locally as '{csv_filename}'")

# 4. Upload the CSV file to Google Drive
# Mount Google Drive if not already mounted
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Define the destination folder in your Google Drive (reusing the Model_Checkpoints folder)
# You can change 'MyDrive/Model_Checkpoints' to your preferred folder path in Drive
drive_destination_folder = '/content/drive/MyDrive/Model_Checkpoints'

# Create the destination folder in Drive if it doesn't exist
os.makedirs(drive_destination_folder, exist_ok=True)

# Define the full destination path for the CSV file in Drive
destination_csv_file = os.path.join(drive_destination_folder, os.path.basename(csv_filename))

# Copy the CSV file to Google Drive
try:
    shutil.copy(csv_filename, destination_csv_file)
    print(f"File '{csv_filename}' berhasil disimpan ke Google Drive di: {destination_csv_file}")
except Exception as e:
    print(f"Gagal menyimpan file '{csv_filename}' ke Google Drive: {e}")

print("\n--- Bagaimana Cara Mengunduh File dari Google Drive: ---")
print("Setelah kode di atas dieksekusi, file `.csv` Anda akan tersimpan di Google Drive. Berikut langkah-langkah untuk mengunduhnya:")
print("1. Buka Google Drive: Pergi ke [Google Drive Anda](https://drive.google.com/).")
print("2. Navigasi ke Folder: Cari folder `Model_Checkpoints` (atau nama folder lain yang Anda tentukan di kode di atas) di dalam `My Drive`.")
print("3. Temukan File: Di dalam folder tersebut, Anda akan menemukan file `training_metrics.csv`.")
print("4. Unduh: Klik kanan pada file tersebut, lalu pilih `Unduh`.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from google.colab import drive

# Mount Google Drive if not already mounted
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Define the path to the metrics CSV file in Google Drive
drive_metrics_path = '/content/drive/MyDrive/Model_Checkpoints/training_metrics.csv'

# Check if the file exists before attempting to load
if os.path.exists(drive_metrics_path):
    # Load the training metrics from the CSV file
    metrics_df = pd.read_csv(drive_metrics_path)

    print("Training metrics loaded successfully from Google Drive.")
    print(metrics_df.head())

    # Plotting Loss Curves
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1) # 1 row, 2 columns, first plot
    sns.lineplot(data=metrics_df, x='epoch', y='train_loss', label='Train Loss', marker='o')
    sns.lineplot(data=metrics_df, x='epoch', y='val_loss', label='Validation Loss', marker='o')
    plt.title('Training and Validation Loss Curve')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(True)
    plt.legend()

    # Plotting Accuracy Curves
    plt.subplot(1, 2, 2) # 1 row, 2 columns, second plot
    sns.lineplot(data=metrics_df, x='epoch', y='train_accuracy', label='Train Accuracy', marker='o')
    sns.lineplot(data=metrics_df, x='epoch', y='val_accuracy', label='Validation Accuracy', marker='o')
    plt.title('Training and Validation Accuracy Curve')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.grid(True)
    plt.legend()

    plt.tight_layout()
    plt.show()

else:
    print(f"Error: File not found at {drive_metrics_path}")
    print("Please ensure 'training_metrics.csv' is saved in your Google Drive 'Model_Checkpoints' folder.")


##Test Model

In [ ]:
import torch
import torch.nn as nn
import timm
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Memastikan model dan device sudah terdefinisi dari sel sebelumnya
# Jika belum, pastikan sel 'Load Model' (qLMKCjqOAc95) sudah dijalankan
# dan variabel 'model', 'device', 'classes' tersedia.

# 1. Muat kembali model yang telah dilatih
model_save_path = "resnet50_best_early_stop_model.pth" # Updated to load the best early stop model

# Inisialisasi ulang model (struktural) yang sama seperti saat training
# Pastikan `num_classes` dan `device` sudah didefinisikan sebelumnya.
# `classes` variable is assumed to be available globally.
num_classes = len(classes)
loaded_model = timm.create_model('resnet50', pretrained=False) # No pretrained weights for loading state_dict
loaded_model.fc = nn.Linear(loaded_model.fc.in_features, num_classes)
loaded_model = loaded_model.to(device)

# Muat state_dict dari file yang disimpan
loaded_model.load_state_dict(torch.load(model_save_path))
loaded_model.eval() # Set model ke mode evaluasi

print(f"Model '{model_save_path}' berhasil dimuat dan siap dievaluasi.")

# 2. Lakukan evaluasi pada seluruh test_loader
all_preds = []
all_labels = []

print("\nMelakukan prediksi pada data pengujian...")
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = loaded_model(imgs)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# 3. Hitung metrik evaluasi
print("\nMenghitung metrik evaluasi...")

# Akurasi
accuracy = accuracy_score(all_labels, all_preds)
print(f"Akurasi Model pada Test Set: {accuracy:.4f}\n")

# Classification Report (Presisi, Recall, F1-score)
print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=classes))

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.xlabel('Prediksi Label')
plt.ylabel('Label Sebenarnya')
plt.title('Confusion Matrix')
plt.show()

print("Evaluasi model selesai.")